In [ ]:
import csv #biblioteca para leitura de csv
import difflib # 
import re #tokentização esta sendo feito por meio do re
import nltk #serve para o mesmo proposito usando word_tokenize e sent_tokenize  
from unidecode import unidecode
from nltk.tokenize import word_tokenize

#Pré processamento para a IA conseguir apenas as palavras chaves 
def pre_processamento(texto):
  
    # seleciona apenas letras e coloca todas em minúsculo 
    #efetua o processo de filtragem, de letras de A a Z, com acentuação e de numero de 1 a 9
    letras_min =  re.findall(r'\b[A-zÀ-úü0-9]+\b', texto.lower())
    print(letras_min)

    #tokens_nltk = word_tokenize(texto.lower(), language = 'portuguese')
    #print(tokens_nltk)

    # remove stopwords
    # remove complementos dentro do portugues, como e, de, para, com, um, uma etc...
    stopwords = nltk.corpus.stopwords.words('portuguese')

    #minha_stopwords = ['um', 'uma', 'varios'] #lista personalizavel
    
    #converter em uma lista as mesmas palavras não usadas anteriores
    stop = set(stopwords)

    #verificação da palavras como uma lista de compreenção
    sem_stopwords = [w for w in letras_min if w not in stop]

    # juntando os tokens novamente em formato de texto e isso precisa mudar
    texto_limpo = " ".join(sem_stopwords)
    print(texto_limpo)
    return texto_limpo

#função para abrir o arquivo de iniciação e aplicar tudo dentro de um dicionario respostas={}
def carregar_base(arquivo): #função para carregar e abrir o arquivo
    respostas = {} #toda resposta vai virar parte do dicionario

    with open(arquivo, 'r', encoding='utf-8') as n: #abre o arquivo que precisa e fecha quando solicitado ou solucionado
        leitor = csv.DictReader(n) #cada linha vira um dicionario pergunta: "oi"; resposta: "tudo bem?"

        for linha in leitor: #vai fazer a leitura de cada linha
            # remover os espaços extras e transformar todas as letras em lowerscale (minusculo)
            pergunta = linha['pergunta'].strip().lower() 
            resposta = linha['resposta'].strip()
            respostas[pergunta] = resposta #copula como um dicionario
    #print(respostas) de teste
    return respostas

#Mudanças na estrutura do arquivo, tentar usar o pandas ao invés do csv
def teste_pandas(): #função para carregar e abrir o arquivo
# 1. Definir o nome do arquivo CSV
    file_name = 'data/problemas.csv'

    with open(arquivo, 'r', encoding='utf-8') as n: 
        leitor = csv.DictReader(n) #

        # 3. Definir a entrada do usuário (conforme solicitado: "oi")
        user_input = "meu site apresenta erro 500"
    
    # (Opcional) Se quiser que o usuário digite no console, troque a linha acima por esta:
    # user_input = input("Digite sua pergunta: ")

    # 4. Procurar a entrada na coluna 'pergunta'
    # Usamos 'str.contains()' para verificar se a entrada está *dentro* do texto da coluna.
    # 'case=False' ignora diferenças entre maiúsculas e minúsculas (ex: "Oi" ou "oi").
    # 'na=False' trata células vazias (NaN) como se não tivessem a palavra.
    
        mask = df['pergunta'].str.contains(user_input, case=False, na=False)
    
    # 5. Filtrar o DataFrame para obter apenas as linhas que correspondem
        result_df = df[mask]

    # 6. Obter e imprimir a resposta
        if not result_df.empty:
        # Pega a resposta da *primeira* correspondência encontrada
            response = result_df.iloc[0]['respostas']
        
            print(f"Entrada: '{user_input}'")
            print(f"Resposta: {response}")
        else:
        # Se o DataFrame 'result_df' estiver vazio (nenhuma correspondência)
            print(f"Entrada: '{user_input}'")
            print("Resposta: Desculpe, não encontrei uma resposta para isso.")
        
    except FileNotFoundError:
        print(f"Erro: O arquivo '{file_name}' não foi encontrado.")
        print("Por favor, verifique se 'iniciacao.csv' está na mesma pasta do seu notebook.")

#detectar a intenção e padrão de comunicação do usuario com o regex
def detectar_intencao(mensagem):
    mensagem_limpa = unidecode(mensagem.lower().strip())
    
    # Padrões regex para detectar intenções
    padroes = {
        'saudacao': [
        r'^o+i+!*$',                     # oi, oii, oiii, oi!
        r'^o+l+a+!*$',                   # ola, olaa, ola!
        r'^(e+\s*)?a+e+!*$',             # ae, e ae, e aeee!
        r'^o+p+a+!*$',                   # opa, opaa
        r'^f+a+l+a+!*$',                 # fala, falaa
        r'^(hey|hello|hi)+!*$',          # hey, hello, hi
        r'^iai+!*$',                     # iai, iaii
        r'^[bs]om\s*dia!*$',             # bom dia, bom dia!
        r'^[bs]oa\s*tarde!*$',           # boa tarde
        r'^[bs]oa\s*noite!*$'            # boa noite
        r'^salve+!*$',                   # salve, salvee
        r'^e?ai+\s*(mano|cara|gente)?!*$', # eai, e aí mano, eai gente
        r'^co?e+?!*$',                   # coé, coee
        r'^bele+za+\s*(mano|cara)?!*$',   # beleza, beleza mano
        r'^como\s*v[aã]o?\s*as\s*coisas.*$', # como vão as coisas
        r'^tudo\s*sussa+!*$',            # tudo sussa, sussa
    ],

    'despedida': [
        r'^t+c+h+a+u+!*$',               # tchau, tchauu
        r'^a+t+[ée]\s*(l[oó]g[o0]|m[aá]is|j[aá])!*$', # até logo, até mais, até já
        r'^f+l+w+!*$',                   # flw, flww
        r'^(bye|goodbye)+!*$',           # bye, goodbye
        r'^v+a+l+e+u+!*$'                # valeu, valeeu
        r'^f+ui+!*$',                    # fui, fuii
        r'^te?\s*v[eê]jo\s*d+ep+oi+s!*$', # te vejo depois
        r'^b+e+i+j+o+s!*$',              # beijos
        r'^ab+c+io\s*e\s*fui!*$',        # abç e fui
        r'^f+al+ou+!*$',                 # falou, falow
    ],

    'nome': [
        r'.*[qk]ual\s*[ée]\s*seu\s*nome.*',
        r'.*[ck]omo\s*vo[cc][ea]\s*se\s*chama.*',
        r'.*seu\s*nome.*',
        r'.*nome.*vo[cc][ea].*',
        r'.*(quem\s*[ée]\s*vo[cc][ea]).*', # quem é você
        r'.*(qual\s*[ée]\s*o\s*seu\s*nome\s*mesmo).*', # qual é o seu nome mesmo
        r'.*(tem\s*nome).*',              # você tem nome?
        r'.*(como\s*posso\s*te\s*chamar).*' # como posso te chamar
    ],

    'tudo_bem': [
        r'.*tudo\s*[bc]em.*',
        r'.*como\s*vo[cc][ea]\s*esta.*',
        r'.*como\s*vo[cc][ea]\s*t[aá].*',
        r'^[bc]e?le?za+!*$',                
        r'^s+u+s+s+a+!*$',                # sussa
        r'^d+e+\s*b+o+a+!*$',             # de boa, de boa?
        r'^t+r+a+n+q+u+i+l+o+!*$',        # tranquilo
        r'^s+u+a+v+e+!*$',                # suave, suavão
        r'^f+irm+a+!*$',                  # firma, firmão
        r'^d+e\s*lev+e+!*$',              # de leve
        r'^na\s*p+a+z+!*$',               # na paz
    ]
}
    
    # Verifica cada categoria
    for intencao, lista_padroes in padroes.items():
        for padrao in lista_padroes:
            if re.search(padrao, mensagem_limpa, re.IGNORECASE):
                return intencao
    
    return "outro"

#função para responder de acordo com a pergunta
def responder(mensagem): 
    respostas = carregar_base("data/iniciacao.csv")

    #tokentização incompleta ainda
    texto = pre_processamento(mensagem)

    # Primeiro tenta detectar a intenção com regex
    intencao = detectar_intencao(texto)
    
    # Mapeia a intenção para a resposta correspondente
    # comentario.: ja temos um função carregar_repostas que retornar a mesma informação e ele puxo tudo do arquivo, poderiamos apenas pegar o retorno da função para usar como mapeamento
    mapeamento_respostas = {
        'saudacao': respostas.get('oi', 'Olá! Tudo bem?'),
        'despedida': respostas.get('tchau', 'Até logo! Volte sempre!'),
        'nome': respostas.get('qual e o seu nome', 'Eu sou um chatbot feito em Python!'),
        'tudo_bem': respostas.get('tudo bem', 'Que bom! Como posso te ajudar?')
    }

    #print(mapeamento_respostas) de teste
    
    # Se encontrou uma intenção conhecida, retorna a resposta correspondente
    # comentario.: se isso tem um if para mapear respostas, poderia ter um elif para procurar os problemas tambem e por um um else para caso não tenha nada na base de dados
    if intencao in mapeamento_respostas:
        return mapeamento_respostas[intencao]
    
    # Se não encontrou por regex, usa o sistema antigo de similaridade
    # comentario.: se conseguir mesclar o regex com o codigo legado, daqui para baixo não precisa mais usar
    mensagem_limpa = unidecode(mensagem.lower().strip())
    perguntas = list(respostas.keys())
    
    # Busca exata
    if mensagem_limpa in perguntas:
        return respostas[mensagem_limpa]
    
    # Busca por similaridade
    parecidas = difflib.get_close_matches(mensagem_limpa, perguntas, n=1, cutoff=0.6) #se tornou obsoleto
    
    if parecidas:
        return respostas[parecidas[0]]
    else:
        return "Desculpe, não entendi o que você quis dizer."



In [35]:
# Carrega as respostas do arquivo CSV
#respostas = carregar_base("data/iniciacao.csv")
#problemas = carregar_problemas("data/problemas.csv") ainda esta incompleto, mas o escopo inicial foi aplicado

#vieira precisa fazer o codigo carregar os arquivo apenas quando for soliciado de acordo com a pergunta
    
print("ChatBot: Olá! Digite 'sair' para encerrar.\n")
    
while True:
    usuario = input("Você: ")
    if usuario.lower() == "sair":
        print("ChatBot: Até mais!")
        break
    print("ChatBot:", responder(usuario))

ChatBot: Olá! Digite 'sair' para encerrar.

--- DataFrame (CSV) Carregado ---
                                             pergunta  \
0                       meu site apresenta não seguro   
1             como liberar meu dominio na registro.br   
2         Como configuro meu email locaweb no outlook   
3   Meu site está fora do ar. Quais são as causas ...   
4   Como faço para acessar ou subir arquivos por F...   
5   Meu site está lento. Como otimizar o desempenh...   
6   O que significa erro 500 Internal Server Error...   
7   Como devo criar as entradas de DNS na Zona de ...   
8   O que fazer quando o acesso ao site apresenta ...   
9    Como instalar o WordPress na Hospedagem Locaweb?   
10  O que é e como emitir o certificado Let's Encr...   
11  Como restaurar um backup da minha Hospedagem d...   
12  O site caiu por excesso de tráfego. Como verif...   
13  Como alterar os servidores DNS de um domínio r...   
14  Qual a diferença entre registrar um domínio na...   
15  O que 

'\nwhile True:\n    usuario = input("Você: ")\n    if usuario.lower() == "sair":\n        print("ChatBot: Até mais!")\n        break\n    print("ChatBot:", responder(usuario))\n'